In [ ]:

# 1. Import Libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import zscore

pd.set_option("display.max_columns", None)
sns.set_style("whitegrid")



# 2. Load Dataset

country = "ethiopia"

df = pd.read_csv(f"../data/{country}.csv")

# Add country column
df["Country"] = "Ethiopia"

print("Initial Shape:", df.shape)
df.head()



# 3. Convert Date

# YEAR + DOY (day of year) → actual date
df["Date"] = pd.to_datetime(df["YEAR"] * 1000 + df["DOY"], format="%Y%j")

# Extract month and year
df["Month"] = df["Date"].dt.month
df["Year"] = df["Date"].dt.year



# 4. Replace Missing Values (-999)

# NASA uses -999 for missing data
df.replace(-999, np.nan, inplace=True)



# 5. Remove Duplicates

duplicate_count = df.duplicated().sum()
print("Duplicate rows found:", duplicate_count)

df = df.drop_duplicates()
print("Shape after removing duplicates:", df.shape)



# 6. Summary Statistics

print("\nSummary Statistics:")
print(df.describe())



# 7. Missing Value Report

missing = df.isna().sum()
missing_percent = (missing / len(df)) * 100

missing_report = pd.DataFrame({
    "Missing": missing,
    "Percent": missing_percent
}).sort_values(by="Percent", ascending=False)

print("\nMissing Value Report:")
print(missing_report)

print("\nColumns with >5% missing:")
print(missing_report[missing_report["Percent"] > 5])



# 8. Outlier Detection (Z-score)

cols = [
    "T2M", "T2M_MAX", "T2M_MIN",
    "PRECTOTCORR", "RH2M",
    "WS2M", "WS2M_MAX"
]

available_cols = [c for c in cols if c in df.columns]

z_scores = np.abs(df[available_cols].apply(zscore, nan_policy="omit"))

outliers = (z_scores > 3).sum()

print("\nOutliers per column:")
print(outliers)

rows_with_outliers = (z_scores > 3).any(axis=1).sum()
print("\nRows with at least one outlier:", rows_with_outliers)



# 9. Handle Missing Values


# Drop rows where more than 30% is missing
rows_before = len(df)
df = df[df.isna().mean(axis=1) < 0.3]
rows_after = len(df)

print("\nRows dropped (>30% missing):", rows_before - rows_after)

# Forward fill for weather columns
df[available_cols] = df[available_cols].ffill()



# 10. Create Temperature Range

if "T2M_MAX" in df.columns and "T2M_MIN" in df.columns:
    df["T2M_RANGE"] = df["T2M_MAX"] - df["T2M_MIN"]



# 11. Export Clean Data

df.to_csv(f"../data/{country}_clean.csv", index=False)
print("\nCleaned dataset saved.")



# 12. Monthly Avg Temperature

monthly_temp = (
    df.set_index("Date")
      .resample("M")["T2M"]
      .mean()
)

warmest = monthly_temp.idxmax()
coolest = monthly_temp.idxmin()

plt.figure(figsize=(12,5))
plt.plot(monthly_temp, linewidth=2)

plt.scatter(warmest, monthly_temp.loc[warmest])
plt.scatter(coolest, monthly_temp.loc[coolest])

plt.title("Monthly Average Temperature")
plt.xlabel("Date")
plt.ylabel("Temperature (T2M)")
plt.show()

print("Warmest Month:", warmest.strftime("%Y-%m"))
print("Coolest Month:", coolest.strftime("%Y-%m"))



# 13. Monthly Rainfall

monthly_rain = (
    df.set_index("Date")
      .resample("M")["PRECTOTCORR"]
      .sum()
)

peak_rain = monthly_rain.idxmax()

plt.figure(figsize=(12,5))
plt.bar(monthly_rain.index, monthly_rain.values)

plt.title("Monthly Rainfall")
plt.xlabel("Date")
plt.ylabel("Rainfall")
plt.show()

print("Peak Rain Month:", peak_rain.strftime("%Y-%m"))



# 14. Correlation Heatmap

corr = df.select_dtypes(include=[np.number]).corr()

plt.figure(figsize=(10,8))
sns.heatmap(corr, cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()



# 15. Strongest Correlations

corr_pairs = corr.unstack().reset_index()
corr_pairs.columns = ["Var1", "Var2", "Corr"]

corr_pairs = corr_pairs[corr_pairs["Var1"] != corr_pairs["Var2"]]

corr_pairs["abs"] = corr_pairs["Corr"].abs()

strongest = corr_pairs.sort_values("abs", ascending=False).drop_duplicates("abs").head(3)

print("\nStrongest Correlations:")
print(strongest[["Var1", "Var2", "Corr"]])



# 16. Scatter Plots

plt.figure(figsize=(6,5))
plt.scatter(df["T2M"], df["RH2M"], alpha=0.5)
plt.xlabel("Temperature")
plt.ylabel("Humidity")
plt.title("Temp vs Humidity")
plt.show()


plt.figure(figsize=(6,5))
plt.scatter(df["T2M_RANGE"], df["WS2M"], alpha=0.5)
plt.xlabel("Temp Range")
plt.ylabel("Wind Speed")
plt.title("Temp Range vs Wind")
plt.show()



# 17. Rainfall Distribution

plt.figure(figsize=(8,5))
plt.hist(df["PRECTOTCORR"].dropna(), bins=50)
plt.yscale("log")
plt.title("Rainfall Distribution")
plt.show()



# 18. Bubble Chart

plt.figure(figsize=(8,6))
plt.scatter(
    df["T2M"],
    df["RH2M"],
    s=df["PRECTOTCORR"]*5 + 5,
    alpha=0.4
)

plt.xlabel("Temperature")
plt.ylabel("Humidity")
plt.title("Temp vs Humidity (Bubble = Rainfall)")
plt.show()


